# INSTALL LIBS

In [4]:
# install libs
!pip install -q \
    "unsloth[colab-new]" \
    datasets \
    trl \
    accelerate \
    bitsandbytes \
    peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 12.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 110.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 79.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 25.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.3 MB/s eta 0:00:00:00:

# IMPORT LIBS

In [13]:
# Imports 
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel

from trl import SFTTrainer
from transformers import TrainingArguments


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import kagglehub

# LOAD MODEL

In [ ]:
# Load model

model_name = "unsloth/gemma-3-4B-it"


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    
    max_seq_length=512,

    load_in_4bit=True,

    dtype=None,
)

# GET PEFT MODEL

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,

    r=16,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",

        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    lora_alpha=16,

    lora_dropout=0,

    bias="none",

    use_gradient_checkpointing="unsloth",

    random_state=42,
)

# LOAD DATASET

In [14]:
dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    split="train"
)


dataset

README.md: 0.00B [00:00, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 26872
})

In [ ]:
dataset[0]

# FORMATTING FUNC

In [ ]:
def formatting_func(example):

    messages = [
        {
            "role": "system",
            "content": 
            "You are a helpful customer support assistant."
        },

        {
            "role": "user",
            "content": example["instruction"]
        },

        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )
    }

dataset = dataset.map(
    formatting_func,
    remove_columns=dataset.column_names
)

# SPLIT DATA

In [17]:
dataset = dataset.train_test_split(
    test_size=0.05,
    seed=42
)


train_dataset = dataset["train"]

eval_dataset = dataset["test"]

# TRAINING ARGS

In [ ]:
training_args = TrainingArguments(

    output_dir="./gemma3-support",

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    warmup_steps=50,

    num_train_epochs=1,

    learning_rate=2e-4,

    fp16=True,
    logging_strategy="steps",
    logging_steps=10,

    eval_strategy="steps",

    eval_steps=100,
    save_strategy="no",

    save_total_limit=2,

    optim="adamw_8bit",

    weight_decay=0.01,

    lr_scheduler_type="cosine",
    disable_tqdm=False,
    report_to="none",
    average_tokens_across_devices=False
)

# SFT TRAINER

In [ ]:
trainer = SFTTrainer(

    model=model,

    tokenizer=tokenizer,

    train_dataset=train_dataset,

    eval_dataset=eval_dataset,

    dataset_text_field="text",

    max_seq_length=512,
    packing = True,

    args=training_args,
)

In [ ]:
lengths = [len(tokenizer(x)["input_ids"][0]) for x in train_dataset["text"][:5000]]
print(max(lengths), sum(lengths)/len(lengths))

In [ ]:
print(max(lengths))

# RUN TRAINING

In [ ]:
from transformers.utils import logging

logging.set_verbosity_info()
trainer.train()

# SAVE MODEL

In [ ]:
model.save_pretrained(
    "gemma3-bitext-support-lora"
)

tokenizer.save_pretrained(
    "gemma3-bitext-support-lora"
)

# EVALUATION

In [ ]:
!kaggle kernels output devendrajadhav2470/fine-tuning-project -p /kaggle/output

In [74]:
!ls /kaggle/output

fine-tuning-project.log  gemma3-bitext-support-lora  unsloth_compiled_cache


## LOAD FINETUNED MODEL

In [ ]:
# COMPARISON

import torch
from unsloth import FastLanguageModel

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-4B-it",
    max_seq_length=512,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(base_model)

In [ ]:
finetuned_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/output/gemma3-bitext-support-lora",
    max_seq_length=512,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(finetuned_model)

## GENERATE RESPONSE

In [10]:
def generate_response(model, question):
    messages = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": "You are a helpful customer support assistant."}
            ]
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": question}
            ]
        }
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
    )


    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

## COMPARING RESPONSES FINETUNED VS BASE

In [11]:
questions = [
    "I accidentally placed an order and want to cancel it",
    "My package has not arrived yet",
    "I was charged twice for the same purchase",
    "How can I change my shipping address?",
    "I forgot my account password"
]

for q in questions:

    print("="*80)
    print("QUESTION:")
    print(q)

    print("\nBASE GEMMA:")
    print(
        generate_response(
            base_model,
            q
        )
    )

    print("\nFINE-TUNED GEMMA:")
    print(
        generate_response(
            finetuned_model,
            q
        )
    )

QUESTION:
I accidentally placed an order and want to cancel it

BASE GEMMA:
Okay, I understand you accidentally placed an order and want to cancel it. I can definitely help you with that! 

To best assist you, I need a little more information. Could you please provide me with one or more of the following:

*   **Order Number:** This is the most important thing! It’s usually a long string of numbers and letters. You can find it in the confirmation email you received after placing the order.
*   **Email Address used for the order:**  This will help me locate the order quickly.
*   **Name used on the order:**  Just the name the order was placed under.

**Important Note:**  Cancellation policies vary depending on the company and the type of product

FINE-TUNED GEMMA:
I understand your situation and the urgency to cancel your order. Rest assured, I'm here to assist you every step of the way. To proceed with the cancellation, please provide me with the order number, as it will help me locate

# ROUGE/BLEU 

In [27]:
!pip install -q evaluate rouge_score sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.8 MB/s eta 0:00:00


In [28]:
eval_samples = eval_dataset.select(range(100))

In [29]:
import evaluate

rouge = evaluate.load("rouge")

In [40]:
def generate_batch(model, questions, batch_size=8):
    all_outputs = []
    for i in tqdm(range(0, len(questions), batch_size)):
        print(f"batch{i+1}")
        batch = questions[i:i+batch_size]
        conversations = [
            [
                {
                    "role": "system",
                    "content": [
                        {"type": "text", "text": "You are a helpful customer support assistant."}
                    ]
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": q}
                    ]
                }
            ]
            for q in batch
        ]
        inputs = tokenizer.apply_chat_template(
            conversations,
            tokenize=True,
            add_generation_prompt=True,
            padding=True,
            return_tensors="pt",
            return_dict=True,
        ).to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

        input_lengths = inputs["attention_mask"].sum(dim=1)

        for output, in_len in zip(outputs, input_lengths):
            text = tokenizer.decode(
                output[in_len:],
                skip_special_tokens=True
            )
            all_outputs.append(text)

    return all_outputs

In [32]:
from tqdm.auto import tqdm

In [41]:
base_predictions = generate_batch(
    base_model,
    [x["instruction"] for x in eval_samples],
    batch_size=8
)

  0%|          | 0/13 [00:00<?, ?it/s]

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


batch1


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


batch9


...

batch81


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


batch89


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


batch97


## ROUGE SCORE FOR BASE MODEL

In [42]:
base_rouge = rouge.compute(
    predictions=base_predictions,
    references=[example["response"] for example in eval_samples]
)

base_rouge

{'rouge1': np.float64(0.3421730967010105),
 'rouge2': np.float64(0.08181139746183233),
 'rougeL': np.float64(0.18861050550709432),
 'rougeLsum': np.float64(0.25398389481014993)}

In [43]:
finetuned_predictions = generate_batch(
    finetuned_model,
    [x["instruction"] for x in eval_samples],
    batch_size=8
)

  0%|          | 0/13 [00:00<?, ?it/s]

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


batch1


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


batch9


...

batch81


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


batch89


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


batch97


## ROUGE SCORE FOR FINETUNED MODEL

In [47]:
finetuned_rouge = rouge.compute(
    predictions=finetuned_predictions,
    references=[example["response"] for example in eval_samples]
)


finetuned_rouge

{'rouge1': np.float64(0.5552683339789664),
 'rouge2': np.float64(0.3159994739432457),
 'rougeL': np.float64(0.4222502620803771),
 'rougeLsum': np.float64(0.4554751374746279)}

## ROUGE COMPARISON DF

In [48]:
import pandas as pd


comparison = pd.DataFrame(
    {
        "Metric": [
            "ROUGE-1",
            "ROUGE-2",
            "ROUGE-L"
        ],

        "Base Gemma": [
            base_rouge["rouge1"],
            base_rouge["rouge2"],
            base_rouge["rougeL"]
        ],

        "Fine-tuned Gemma": [
            finetuned_rouge["rouge1"],
            finetuned_rouge["rouge2"],
            finetuned_rouge["rougeL"]
        ]
    }
)


comparison

,Metric,Base Gemma,Fine-tuned Gemma
0,ROUGE-1,0.342173,0.555268
1,ROUGE-2,0.081811,0.315999
2,ROUGE-L,0.188611,0.422250


In [54]:
finetuned_score = finetuned_rouge["rougeL"]
base_score = base_rouge["rougeL"]

# unwrap if using rouge_score's AggregateScore objects
if hasattr(finetuned_score, "mid"):
    finetuned_score = finetuned_score.mid.fmeasure
    base_score = base_score.mid.fmeasure

if base_score == 0:
    print("base_rouge['rougeL'] is 0 — percent change undefined")
else:
    pct_change = ((finetuned_score - base_score) / base_score) * 100
    print(f"{pct_change:.2f}%")

123.87%


# BERT

In [68]:
import gc
import torch

# Delete any large models you loaded earlier.
# Replace these names with your actual variables.
for variable_name in ["model", "processor", "tokenizer", "vlm_model"]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print(
    f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)
print(
    f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

Allocated: 9.81 GB
Reserved:  9.97 GB


In [61]:
import torch, gc

# delete references to previously loaded models
del base_model
del finetuned_model  # if it exists in your notebook
gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")

10.312498176 GB allocated
10.408165376 GB reserved


In [66]:
!pip install -q bert-score

In [57]:
references=[example["response"] for example in eval_samples]

## BERTSCORE FOR BASE

In [69]:
from bert_score import score

# Compute BERTScore
P, R, F1 = score(
    base_predictions,
    references,
    lang="en",      # Language
    verbose=True
)

print(f"Average Precision: {P.mean().item():.4f}")
print(f"Average Recall:    {R.mean().item():.4f}")
print(f"Average F1:        {F1.mean().item():.4f}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 7.75 seconds, 12.91 sentences/sec
Average Precision: 0.8289
Average Recall:    0.8583
Average F1:        0.8431


## BERTSCORE FOR FINETUNED

In [71]:
from bert_score import score

# Compute BERTScore
P, R, F1 = score(
    finetuned_predictions,
    references,
    lang="en",      # Language
    verbose=True
)

print(f"Average Precision: {P.mean().item():.4f}")
print(f"Average Recall:    {R.mean().item():.4f}")
print(f"Average F1:        {F1.mean().item():.4f}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 6.55 seconds, 15.26 sentences/sec
Average Precision: 0.9178
Average Recall:    0.9080
Average F1:        0.9127


# PUSH TO HF

In [72]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

import os
from huggingface_hub import login

login(secret_value_0)

In [75]:
!ls /kaggle/output

fine-tuning-project.log  gemma3-bitext-support-lora  unsloth_compiled_cache


In [77]:
from huggingface_hub import login, HfApi, create_repo

api = HfApi()

create_repo(
    repo_id="devendrajadhav34/gemma3-bitext-support-lora",
    repo_type="model",
    private=False,  # set True if you want it private
    exist_ok=True,  # won't error if it already exists
)

api.upload_folder(
    folder_path="/kaggle/output/gemma3-bitext-support-lora",
    repo_id="devendrajadhav34/gemma3-bitext-support-lora",
    repo_type="model",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/devendrajadhav34/gemma3-bitext-support-lora/commit/4aa892e209932944ca34a9239802d2b0503b7c1a', commit_message='Upload folder using huggingface_hub', commit_description='', oid='4aa892e209932944ca34a9239802d2b0503b7c1a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/devendrajadhav34/gemma3-bitext-support-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='devendrajadhav34/gemma3-bitext-support-lora'), pr_revision=None, pr_num=None)

In [79]:
import os

print(os.listdir("/kaggle/output/gemma3-bitext-support-lora"))

['tokenizer.json', 'adapter_config.json', 'tokenizer_config.json', 'chat_template.jinja', 'tokenizer.model', 'README.md', 'processor_config.json', 'adapter_model.safetensors']


# GRADIO DEMO

In [2]:
!pip install -q gradio

In [ ]:
# devendrajadhav34/gemma3-bitext-support-lora

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="devendrajadhav34/gemma3-bitext-support-lora",
    max_seq_length=512,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

In [16]:
def generate_response(question):
    messages = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": "You are a helpful customer support assistant."}
            ]
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": question}
            ]
        }
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
    )


    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

In [17]:
print(generate_response("Hi, my order cancel?"))

I'll do my best! I understand your concern about canceling your order. To help you with this, could you please provide me with the order details, such as the order number or any other relevant information? This will enable me to assist you more effectively and ensure that your cancellation is handled promptly. Rest assured, I'm here to support you every step of the way.


In [18]:
import gradio as gr

demo = gr.Interface(
    fn=generate_response,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Describe your customer support issue...",
        label="Customer Query",
    ),
    outputs=gr.Textbox(
        lines=10,
        label="Assistant Response",
    ),
    title="Gemma 3 Customer Support Assistant",
    description=(
        "Fine-tuned Gemma 3 4B-IT using QLoRA on the Bitext Customer Support dataset."
    ),
    examples=[
        ["I accidentally placed an order. Can I cancel it?"],
        ["My package hasn't arrived yet."],
        ["I was charged twice for my order."],
        ["How do I reset my password?"],
        ["Can I change my shipping address after placing an order?"],
    ],
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7864
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://2e4fc2fe433f828f18.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
!gradio deploy

Need 'write' access token to create a Spaces repo.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

Enter your token (input will not be visible): 